In [2]:
from collections import Counter

def get_wordle_pattern(guess, answer):
    """
    Generate Wordle feedback pattern.
    Returns tuple: 0=gray (not in word), 1=yellow (wrong position), 2=green (correct)
    """
    result = [0] * 5
    answer_counts = Counter(answer)
    
    # First pass: mark all exact matches (green)
    for i, (g, a) in enumerate(zip(guess, answer)):
        if g == a:
            result[i] = 2
            answer_counts[g] -= 1  # Consume this letter
    
    # Second pass: mark letters in wrong positions (yellow)
    for i, g in enumerate(guess):
        if result[i] == 0 and answer_counts[g] > 0:
            result[i] = 1
            answer_counts[g] -= 1  # Consume this letter
    
    return tuple(result)

def apply_constraints(word, guess, pattern):
    """Check if a word is consistent with observed guess/pattern"""
    return get_wordle_pattern(guess, word) == pattern

# Load word list from NLTK
print("=== Loading Word List ===\n")

try:
    import nltk
    from nltk.corpus import words
    
    # Download if not already present
    try:
        words.words()
    except LookupError:
        print("Downloading NLTK words corpus...")
        nltk.download('words', quiet=True)
    
    # Get all 5-letter words, uppercase
    all_words = words.words()
    WORD_LIST = sorted(set(word.upper() for word in all_words if len(word) == 5 and word.isalpha()))
    
    print(f"Loaded {len(WORD_LIST)} five-letter words from NLTK")
    print(f"Sample words: {WORD_LIST[:10]}")
    print()
    
except ImportError:
    print("NLTK not installed. Using fallback word list.")
    # Fallback to a larger curated list
    WORD_LIST = ["SALET", "ROATE", "RAISE", "ARISE", "IRATE", "CRANE", "SLATE", 
                 "CRATE", "TRACE", "STARE", "ADORE", "ALONE", "ATONE", "STORE",
                 "SHORE", "SNORE", "SPORE", "SCORE", "SWORE", "SHONE", "STONE",
                 "DRONE", "PRONE", "OZONE", "PHONE", "THOSE", "WHOSE", "CHOSE",
                 "BRAKE", "FLAKE", "SHAKE", "SNAKE", "QUAKE", "AWAKE"]
    print(f"Using fallback list: {len(WORD_LIST)} words\n")

# Demonstration
print("=== Wordle Feedback Mechanism ===\n")

test_cases = [
    ("ROBOT", "FLOOR"),  # Repeated letters
    ("CRANE", "REACT"),  # Anagrams
    ("SALET", "LASER"),  # Close match
]

for guess, answer in test_cases:
    pattern = get_wordle_pattern(guess, answer)
    pattern_str = ''.join(['⬜' if p == 0 else '🟨' if p == 1 else '🟩' for p in pattern])
    print(f"Guess: {guess}")
    print(f"Answer: {answer}")
    print(f"Pattern: {pattern_str} {pattern}")
    print()

# Show constraint propagation with realistic word list
print("=== Constraint Propagation Example ===\n")
# Select words starting with 'FL' from full list
sample_words = [w for w in WORD_LIST if w.startswith('FL')][:8]
if len(sample_words) < 5:
    sample_words = WORD_LIST[:8]

guess = "CRANE"
answer = sample_words[0] if sample_words else "FLOUR"
pattern = get_wordle_pattern(guess, answer)

print(f"Sample from {len(WORD_LIST)} total words: {sample_words}")
print(f"After guessing '{guess}' with answer '{answer}':")
print(f"Pattern: {pattern}")
remaining = [w for w in sample_words if apply_constraints(w, guess, pattern)]
print(f"Remaining possibilities: {remaining}")
print(f"Eliminated: {set(sample_words) - set(remaining)}")
print(f"\nFrom full word list, {len([w for w in WORD_LIST if apply_constraints(w, guess, pattern)])} words remain")

=== Loading Word List ===

Loaded 9972 five-letter words from NLTK
Sample words: ['AALII', 'AARON', 'ABACA', 'ABACK', 'ABAFF', 'ABAFT', 'ABAMA', 'ABASE', 'ABASH', 'ABASK']

=== Wordle Feedback Mechanism ===

Guess: ROBOT
Answer: FLOOR
Pattern: 🟨🟨⬜🟩⬜ (1, 1, 0, 2, 0)

Guess: CRANE
Answer: REACT
Pattern: 🟨🟨🟩⬜🟨 (1, 1, 2, 0, 1)

Guess: SALET
Answer: LASER
Pattern: 🟨🟩🟨🟩⬜ (1, 2, 1, 2, 0)

=== Constraint Propagation Example ===

Sample from 9972 total words: ['FLACK', 'FLAFF', 'FLAIL', 'FLAIR', 'FLAKE', 'FLAKY', 'FLAMB', 'FLAME']
After guessing 'CRANE' with answer 'FLACK':
Pattern: (1, 0, 2, 0, 0)
Remaining possibilities: ['FLACK']
Eliminated: {'FLAMB', 'FLAFF', 'FLAIR', 'FLAIL', 'FLAME', 'FLAKE', 'FLAKY'}

From full word list, 49 words remain


In [3]:
from collections import defaultdict, Counter

# Letter frequency data from Wikipedia (English text)
# https://en.wikipedia.org/wiki/Letter_frequency
LETTER_FREQUENCIES = {
    'E': 12.70, 'T': 9.06, 'A': 8.17, 'O': 7.51, 'I': 6.97,
    'N': 6.75, 'S': 6.33, 'H': 6.09, 'R': 5.99, 'D': 4.25,
    'L': 4.03, 'C': 2.78, 'U': 2.76, 'M': 2.41, 'W': 2.36,
    'F': 2.23, 'G': 2.02, 'Y': 1.97, 'P': 1.93, 'B': 1.29,
    'V': 0.98, 'K': 0.77, 'J': 0.15, 'X': 0.15, 'Q': 0.10,
    'Z': 0.07
}

def build_frequency_tables(words):
    """Calculate letter frequencies by position from word list"""
    position_freq = [defaultdict(int) for _ in range(5)]
    
    for word in words:
        for i, letter in enumerate(word):
            position_freq[i][letter] += 1
    
    return position_freq

def score_word_frequency(word, freq_tables, remaining_words):
    """Score word based on letter/position frequencies"""
    # Use position-specific frequencies from the remaining words
    score = sum(freq_tables[i][letter] for i, letter in enumerate(word))
    
    # Bonus for unique letters (more information gathered)
    unique_ratio = len(set(word)) / len(word)
    score *= (1 + unique_ratio)
    
    # Add bonus based on general English letter frequency
    english_freq_score = sum(LETTER_FREQUENCIES.get(letter, 0) for letter in set(word))
    score += english_freq_score * 0.1  # Small weight for general frequency
    
    return score

def frequency_agent_solve(answer, word_list, verbose=True):
    """Solve Wordle using frequency heuristics"""
    remaining = word_list.copy()
    guesses = []
    
    for attempt in range(6):
        freq_tables = build_frequency_tables(remaining)
        
        # Choose word with highest frequency score from remaining possibilities
        best_word = max(remaining, key=lambda w: score_word_frequency(w, freq_tables, remaining))
        guesses.append(best_word)
        
        pattern = get_wordle_pattern(best_word, answer)
        
        if verbose:
            pattern_str = ''.join(['⬜' if p == 0 else '🟨' if p == 1 else '🟩' for p in pattern])
            print(f"Attempt {attempt + 1}: {best_word} -> {pattern_str}")
            print(f"  Remaining words: {len(remaining)}")
        
        if best_word == answer:
            if verbose:
                print(f"✓ Solved in {len(guesses)} guesses!\n")
            return guesses
        
        # Update remaining words based on constraints
        remaining = [w for w in remaining if apply_constraints(w, best_word, pattern)]
        
        if not remaining:
            if verbose:
                print(f"✗ No valid words remain! Answer was {answer}\n")
            return guesses
    
    if verbose:
        print(f"✗ Failed to solve in 6 guesses. Answer was {answer}\n")
    return guesses

# Test the frequency agent
print("=== Frequency-Based Heuristic Agent ===\n")
print(f"Total word list size: {len(WORD_LIST)}\n")

# Select diverse test words
import random
random.seed(42)
test_answers = random.sample(WORD_LIST, 3)

print(f"Testing on random words: {test_answers}\n")

for answer in test_answers:
    print(f"Target word: {answer}")
    result = frequency_agent_solve(answer, WORD_LIST)
    print()

# Show top words by frequency
print("=== Analysis: Best Opening Words by Frequency ===\n")
freq_tables = build_frequency_tables(WORD_LIST)
all_scores = [(w, score_word_frequency(w, freq_tables, WORD_LIST)) for w in WORD_LIST]
all_scores.sort(key=lambda x: x[1], reverse=True)

print("Top 10 words by frequency score:")
for i, (word, score) in enumerate(all_scores[:10], 1):
    unique_letters = len(set(word))
    print(f"{i:2}. {word}: {score:.2f} ({unique_letters} unique letters)")

=== Frequency-Based Heuristic Agent ===

Total word list size: 9972

Testing on random words: ['CIDER', 'AMPLY', 'KETCH']

Target word: CIDER
Attempt 1: SAITE -> ⬜⬜🟨⬜🟨
  Remaining words: 9972
Attempt 2: FINER -> ⬜🟩⬜🟩🟩
  Remaining words: 259
Attempt 3: HIVER -> ⬜🟩⬜🟩🟩
  Remaining words: 39
Attempt 4: MIDER -> ⬜🟩🟩🟩🟩
  Remaining words: 26
Attempt 5: CIDER -> 🟩🟩🟩🟩🟩
  Remaining words: 4
✓ Solved in 5 guesses!


Target word: AMPLY
Attempt 1: SAITE -> ⬜🟨⬜⬜⬜
  Remaining words: 9972
Attempt 2: CORAL -> ⬜⬜⬜🟨🟨
  Remaining words: 822
Attempt 3: PLAGA -> 🟨🟨🟨⬜⬜
  Remaining words: 68
Attempt 4: AMPLY -> 🟩🟩🟩🟩🟩
  Remaining words: 2
✓ Solved in 4 guesses!


Target word: KETCH
Attempt 1: SAITE -> ⬜⬜⬜🟨🟨
  Remaining words: 9972
Attempt 2: TENET -> 🟨🟩⬜⬜⬜
  Remaining words: 226
Attempt 3: KETCH -> 🟩🟩🟩🟩🟩
  Remaining words: 15
✓ Solved in 3 guesses!


=== Analysis: Best Opening Words by Frequency ===

Top 10 words by frequency score:
 1. SAITE: 11550.32 (5 unique letters)
 2. BARIE: 11485.51 (5 unique letters)


In [4]:
import math

def calculate_pattern_entropy(word, possible_words):
    """
    Calculate expected information gain (entropy) for a guess.
    Higher entropy = more expected information gained.
    """
    if not possible_words:
        return 0
    
    # Count how many words produce each pattern
    pattern_counts = {}
    for candidate in possible_words:
        pattern = get_wordle_pattern(word, candidate)
        pattern_counts[pattern] = pattern_counts.get(pattern, 0) + 1
    
    # Calculate entropy: -sum(p * log2(p))
    total = len(possible_words)
    entropy = 0
    for count in pattern_counts.values():
        if count > 0:
            probability = count / total
            entropy -= probability * math.log2(probability)
    
    return entropy

def entropy_agent_solve(answer, word_list, verbose=True):
    """Solve Wordle using information theory (entropy maximization)"""
    remaining = word_list.copy()
    guesses = []
    
    for attempt in range(6):
        # For first guess, we can pre-compute on full list
        # For subsequent guesses, only consider remaining words for efficiency
        if attempt == 0 and len(remaining) > 1000:
            # Sample for efficiency on first guess with large word list
            sample_size = min(2000, len(remaining))
            candidates = random.sample(remaining, sample_size)
        else:
            candidates = remaining
        
        # Calculate entropy for candidate guesses
        word_scores = [(w, calculate_pattern_entropy(w, remaining)) for w in candidates]
        word_scores.sort(key=lambda x: x[1], reverse=True)
        
        best_word = word_scores[0][0]
        best_entropy = word_scores[0][1]
        guesses.append(best_word)
        
        pattern = get_wordle_pattern(best_word, answer)
        
        if verbose:
            pattern_str = ''.join(['⬜' if p == 0 else '🟨' if p == 1 else '🟩' for p in pattern])
            print(f"Attempt {attempt + 1}: {best_word} -> {pattern_str}")
            print(f"  Entropy: {best_entropy:.2f} bits")
            print(f"  Remaining words: {len(remaining)}")
        
        if best_word == answer:
            if verbose:
                print(f"✓ Solved in {len(guesses)} guesses!\n")
            return guesses
        
        remaining = [w for w in remaining if apply_constraints(w, best_word, pattern)]
        
        if not remaining:
            if verbose:
                print(f"✗ No valid words remain! Answer was {answer}\n")
            return guesses
    
    if verbose:
        print(f"✗ Failed to solve in 6 guesses. Answer was {answer}\n")
    return guesses

# Test the entropy agent
print("=== Information Theory (Entropy) Agent ===\n")

# Use same test words for comparison
random.seed(42)
test_answers = random.sample(WORD_LIST, 3)

print(f"Testing on: {test_answers}\n")

for answer in test_answers:
    print(f"Target word: {answer}")
    result = entropy_agent_solve(answer, WORD_LIST)
    print()

# Compare first guesses
print("=== First Guess Comparison ===\n")

# Sample words for entropy calculation (full list too slow)
print("Calculating best opening words (sampling for efficiency)...\n")
sample_words = random.sample(WORD_LIST, min(1000, len(WORD_LIST)))

print("Top 5 words by frequency score:")
freq_tables = build_frequency_tables(WORD_LIST)
freq_scores = [(w, score_word_frequency(w, freq_tables, WORD_LIST)) for w in sample_words]
freq_scores.sort(key=lambda x: x[1], reverse=True)
for word, score in freq_scores[:5]:
    print(f"  {word}: {score:.2f}")

print("\nTop 5 words by entropy:")
entropy_scores = [(w, calculate_pattern_entropy(w, WORD_LIST)) for w in sample_words]
entropy_scores.sort(key=lambda x: x[1], reverse=True)
for word, entropy in entropy_scores[:5]:
    print(f"  {word}: {entropy:.2f} bits")

=== Information Theory (Entropy) Agent ===

Testing on: ['CIDER', 'AMPLY', 'KETCH']

Target word: CIDER
Attempt 1: SERAI -> ⬜🟨🟨⬜🟨
  Entropy: 5.89 bits
  Remaining words: 9972
Attempt 2: DITER -> 🟨🟩⬜🟩🟩
  Entropy: 3.74 bits
  Remaining words: 165
Attempt 3: BIDER -> ⬜🟩🟩🟩🟩
  Entropy: 0.65 bits
  Remaining words: 6
Attempt 4: CIDER -> 🟩🟩🟩🟩🟩
  Entropy: 0.72 bits
  Remaining words: 5
✓ Solved in 4 guesses!


Target word: AMPLY
Attempt 1: TERAS -> ⬜⬜⬜🟨⬜
  Entropy: 5.83 bits
  Remaining words: 9972
Attempt 2: MANIA -> 🟨🟨⬜⬜⬜
  Entropy: 5.45 bits
  Remaining words: 950
Attempt 3: ALBUM -> 🟩🟨⬜⬜🟨
  Entropy: 3.88 bits
  Remaining words: 23
Attempt 4: AMPLY -> 🟩🟩🟩🟩🟩
  Entropy: 1.00 bits
  Remaining words: 2
✓ Solved in 4 guesses!


Target word: KETCH
Attempt 1: SINAE -> ⬜⬜⬜⬜🟨
  Entropy: 5.86 bits
  Remaining words: 9972
Attempt 2: RELET -> ⬜🟩⬜⬜🟨
  Entropy: 5.63 bits
  Remaining words: 699
Attempt 3: HETTY -> 🟨🟩🟩⬜⬜
  Entropy: 3.35 bits
  Remaining words: 20
Attempt 4: FETCH -> ⬜🟩🟩🟩🟩
  Entropy: 0.92 b

In [5]:
def calculate_max_partition(word, possible_words):
    """
    Calculate the size of the largest partition (worst case).
    Lower is better - means we're guaranteed to eliminate more words.
    """
    if not possible_words:
        return 0
    
    pattern_counts = {}
    for candidate in possible_words:
        pattern = get_wordle_pattern(word, candidate)
        pattern_counts[pattern] = pattern_counts.get(pattern, 0) + 1
    
    # Return size of largest partition
    return max(pattern_counts.values()) if pattern_counts else 0

def minimax_agent_solve(answer, word_list, verbose=True):
    """Solve Wordle using minimax (minimize worst-case remaining)"""
    remaining = word_list.copy()
    guesses = []
    
    for attempt in range(6):
        # Sample for efficiency on large word lists
        if len(remaining) > 500:
            candidates = random.sample(remaining, min(500, len(remaining)))
        else:
            candidates = remaining
        
        # Find word that minimizes maximum partition size
        word_scores = [(w, calculate_max_partition(w, remaining)) for w in candidates]
        word_scores.sort(key=lambda x: x[1])  # Lower is better
        
        best_word = word_scores[0][0]
        worst_case = word_scores[0][1]
        guesses.append(best_word)
        
        pattern = get_wordle_pattern(best_word, answer)
        
        if verbose:
            pattern_str = ''.join(['⬜' if p == 0 else '🟨' if p == 1 else '🟩' for p in pattern])
            print(f"Attempt {attempt + 1}: {best_word} -> {pattern_str}")
            print(f"  Worst-case remaining: {worst_case}")
            print(f"  Actual remaining: {len(remaining)}")
        
        if best_word == answer:
            if verbose:
                print(f"✓ Solved in {len(guesses)} guesses!\n")
            return guesses
        
        remaining = [w for w in remaining if apply_constraints(w, best_word, pattern)]
        
        if not remaining:
            if verbose:
                print(f"✗ No valid words remain! Answer was {answer}\n")
            return guesses
    
    if verbose:
        print(f"✗ Failed to solve in 6 guesses. Answer was {answer}\n")
    return guesses

# Test minimax agent
print("=== Minimax (Worst-Case Optimization) Agent ===\n")

random.seed(42)
test_answers = random.sample(WORD_LIST, 3)

print(f"Testing on: {test_answers}\n")

for answer in test_answers:
    print(f"Target word: {answer}")
    result = minimax_agent_solve(answer, WORD_LIST)
    print()

# Strategy comparison on first guess
print("=== Strategy Comparison for First Guess ===\n")
print("Analyzing sample words...\n")

sample_words = random.sample(WORD_LIST, min(200, len(WORD_LIST)))

# Get top words by each metric
entropy_ranked = sorted([(w, calculate_pattern_entropy(w, WORD_LIST)) for w in sample_words],
                        key=lambda x: x[1], reverse=True)
minimax_ranked = sorted([(w, calculate_max_partition(w, WORD_LIST)) for w in sample_words],
                        key=lambda x: x[1])

print("Top 5 by Entropy (maximize information):")
for word, entropy in entropy_ranked[:5]:
    max_part = calculate_max_partition(word, WORD_LIST)
    print(f"  {word}: {entropy:.2f} bits (max partition: {max_part})")

print("\nTop 5 by Minimax (minimize worst case):")
for word, max_part in minimax_ranked[:5]:
    entropy = calculate_pattern_entropy(word, WORD_LIST)
    print(f"  {word}: max partition {max_part} (entropy: {entropy:.2f} bits)")

=== Minimax (Worst-Case Optimization) Agent ===

Testing on: ['CIDER', 'AMPLY', 'KETCH']

Target word: CIDER
Attempt 1: MANEI -> ⬜⬜⬜🟩🟨
  Worst-case remaining: 865
  Actual remaining: 9972
Attempt 2: RIVET -> 🟨🟩⬜🟩⬜
  Worst-case remaining: 34
  Actual remaining: 153
Attempt 3: FIRED -> ⬜🟩🟨🟩🟨
  Worst-case remaining: 15
  Actual remaining: 34
Attempt 4: BIDER -> ⬜🟩🟩🟩🟩
  Worst-case remaining: 4
  Actual remaining: 7
Attempt 5: CIDER -> 🟩🟩🟩🟩🟩
  Worst-case remaining: 3
  Actual remaining: 4
✓ Solved in 5 guesses!


Target word: AMPLY
Attempt 1: INSEA -> ⬜⬜⬜⬜🟨
  Worst-case remaining: 899
  Actual remaining: 9972
Attempt 2: ROYAL -> ⬜⬜🟨🟨🟨
  Worst-case remaining: 80
  Actual remaining: 899
Attempt 3: BALLY -> ⬜🟨⬜🟩🟩
  Worst-case remaining: 9
  Actual remaining: 46
Attempt 4: AMPLY -> 🟩🟩🟩🟩🟩
  Worst-case remaining: 1
  Actual remaining: 4
✓ Solved in 4 guesses!


Target word: KETCH
Attempt 1: NARES -> ⬜⬜⬜🟨⬜
  Worst-case remaining: 789
  Actual remaining: 9972
Attempt 2: TILDE -> 🟨⬜⬜⬜🟨
  Worst-case 

In [6]:
def hybrid_agent_solve(answer, word_list, switch_threshold=5, verbose=True):
    """
    Hybrid strategy: use entropy when many possibilities remain,
    switch to minimax when few remain.
    """
    remaining = word_list.copy()
    guesses = []
    
    for attempt in range(6):
        # Choose strategy based on remaining possibilities
        if len(remaining) > switch_threshold:
            # Use entropy for information gathering
            word_scores = [(w, calculate_pattern_entropy(w, remaining)) for w in remaining]
            word_scores.sort(key=lambda x: x[1], reverse=True)
            strategy_used = "ENTROPY"
        else:
            # Use minimax for guaranteed bounds
            word_scores = [(w, calculate_max_partition(w, remaining)) for w in remaining]
            word_scores.sort(key=lambda x: x[1])  # Lower is better
            strategy_used = "MINIMAX"
        
        best_word = word_scores[0][0]
        guesses.append(best_word)
        
        pattern = get_wordle_pattern(best_word, answer)
        
        if verbose:
            pattern_str = ''.join(['⬜' if p == 0 else '🟨' if p == 1 else '🟩' for p in pattern])
            print(f"Attempt {attempt + 1}: {best_word} -> {pattern_str} [{strategy_used}]")
            print(f"  Remaining words: {len(remaining)}")
        
        if best_word == answer:
            if verbose:
                print(f"✓ Solved in {len(guesses)} guesses!\n")
            return guesses
        
        remaining = [w for w in remaining if apply_constraints(w, best_word, pattern)]
    
    if verbose:
        print(f"✗ Failed to solve. Answer was {answer}\n")
    return guesses

# Test hybrid agent
print("=== Hybrid Strategy Agent ===\n")
print("Strategy: Entropy when >5 words remain, Minimax otherwise\n")

test_answers = ["DOWEL", "GNOME", "FUGUE"]
for answer in test_answers:
    print(f"Target word: {answer}")
    result = hybrid_agent_solve(answer, WORD_LIST, switch_threshold=5)
    print()

=== Hybrid Strategy Agent ===

Strategy: Entropy when >5 words remain, Minimax otherwise

Target word: DOWEL
Attempt 1: TARIE -> ⬜⬜⬜⬜🟨 [ENTROPY]
  Remaining words: 9972
Attempt 2: SOLEN -> ⬜🟩🟨🟩⬜ [ENTROPY]
  Remaining words: 521
Attempt 3: DOWEL -> 🟩🟩🟩🟩🟩 [ENTROPY]
  Remaining words: 13
✓ Solved in 3 guesses!


Target word: GNOME
Attempt 1: TARIE -> ⬜⬜⬜⬜🟩 [ENTROPY]
  Remaining words: 9972
Attempt 2: SOCLE -> ⬜🟨⬜⬜🟩 [ENTROPY]
  Remaining words: 294
Attempt 3: EPODE -> ⬜⬜🟩⬜🟩 [ENTROPY]
  Remaining words: 15
Attempt 4: KNOWE -> ⬜🟩🟩⬜🟩 [MINIMAX]
  Remaining words: 4
Attempt 5: GNOME -> 🟩🟩🟩🟩🟩 [MINIMAX]
  Remaining words: 1
✓ Solved in 5 guesses!


Target word: FUGUE
Attempt 1: TARIE -> ⬜⬜⬜⬜🟩 [ENTROPY]
  Remaining words: 9972
Attempt 2: SOCLE -> ⬜⬜⬜⬜🟩 [ENTROPY]
  Remaining words: 294
Attempt 3: WENDE -> ⬜⬜⬜⬜🟩 [ENTROPY]
  Remaining words: 49
Attempt 4: FUGUE -> 🟩🟩🟩🟩🟩 [MINIMAX]
  Remaining words: 2
✓ Solved in 4 guesses!




In [7]:
import random
from time import time

def run_benchmark(word_list, num_trials=10):
    """Compare all strategies on random sample of words"""
    
    # Select random target words
    random.seed(42)  # For reproducibility
    test_words = random.sample(word_list, min(num_trials, len(word_list)))
    
    agents = {
        'Frequency': frequency_agent_solve,
        'Entropy': entropy_agent_solve,
        'Minimax': minimax_agent_solve,
        'Hybrid': lambda ans, wl, v: hybrid_agent_solve(ans, wl, 50, v)
    }
    
    results = {name: [] for name in agents}
    
    print("=== Performance Benchmark ===\n")
    print(f"Testing {len(test_words)} random words from {len(word_list)} total")
    print(f"Test words: {test_words}\n")
    
    for name, agent_func in agents.items():
        print(f"Testing {name} agent...")
        start_time = time()
        
        for word in test_words:
            guesses = agent_func(word, word_list, False)
            results[name].append(len(guesses))
        
        elapsed = time() - start_time
        
        avg_guesses = sum(results[name]) / len(results[name])
        max_guesses = max(results[name])
        min_guesses = min(results[name])
        
        print(f"  Average: {avg_guesses:.2f} guesses")
        print(f"  Best: {min_guesses}, Worst: {max_guesses}")
        print(f"  Time: {elapsed:.3f}s")
        print()
    
    # Summary comparison
    print("=== Summary Comparison ===\n")
    print(f"{'Strategy':<12} {'Avg':<6} {'Min':<5} {'Max':<5} {'Success%':<10}")
    print("-" * 45)
    for name in agents:
        avg = sum(results[name]) / len(results[name])
        success_rate = sum(1 for g in results[name] if g <= 6) / len(results[name]) * 100
        print(f"{name:<12} {avg:<6.2f} {min(results[name]):<5} {max(results[name]):<5} {success_rate:<10.1f}")
    
    return results

# Run benchmark
print("Note: With a large word list, this may take a minute...\n")
results = run_benchmark(WORD_LIST, num_trials=2)

# Additional analysis
print("\n=== Word List Statistics ===\n")
print(f"Total words: {len(WORD_LIST)}")
print(f"Words with 5 unique letters: {sum(1 for w in WORD_LIST if len(set(w)) == 5)}")
print(f"Words with repeated letters: {sum(1 for w in WORD_LIST if len(set(w)) < 5)}")

# Letter distribution
all_letters = Counter()
for word in WORD_LIST:
    all_letters.update(word)

print(f"\nMost common letters in word list:")
for letter, count in all_letters.most_common(10):
    percentage = (count / (len(WORD_LIST) * 5)) * 100
    wiki_freq = LETTER_FREQUENCIES.get(letter, 0)
    print(f"  {letter}: {count:5} ({percentage:5.2f}% vs {wiki_freq:5.2f}% in English text)")

Note: With a large word list, this may take a minute...

=== Performance Benchmark ===

Testing 2 random words from 9972 total
Test words: ['CIDER', 'AMPLY']

Testing Frequency agent...
  Average: 4.50 guesses
  Best: 4, Worst: 5
  Time: 0.070s

Testing Entropy agent...
  Average: 4.00 guesses
  Best: 4, Worst: 4
  Time: 69.887s

Testing Minimax agent...
  Average: 4.50 guesses
  Best: 4, Worst: 5
  Time: 17.691s

Testing Hybrid agent...
  Average: 4.00 guesses
  Best: 4, Worst: 4
  Time: 339.981s

=== Summary Comparison ===

Strategy     Avg    Min   Max   Success%  
---------------------------------------------
Frequency    4.50   4     5     100.0     
Entropy      4.00   4     4     100.0     
Minimax      4.50   4     5     100.0     
Hybrid       4.00   4     4     100.0     

=== Word List Statistics ===

Total words: 9972
Words with 5 unique letters: 6373
Words with repeated letters: 3599

Most common letters in word list:
  A:  5627 (11.29% vs  8.17% in English text)
  E:  480

In [8]:
# Simulated RL agent using Ollama
# Requires: pip install ollama (or just observe the code structure)

def simulate_rl_agent_with_llm(answer, word_list, verbose=True):
    """
    Simulate an RL agent using an LLM to make strategic decisions.
    The LLM has 'learned' Wordle strategy from its training data.
    """
    try:
        import ollama
    except ImportError:
        print("Ollama not installed. Showing simulation structure only.\n")
        return []
    
    remaining = word_list.copy()
    guesses = []
    history = []
    
    if verbose:
        print("=== Simulated RL Agent (via llama3.2) ===\n")
    
    for attempt in range(6):
        # Build prompt with current game state
        remaining_sample = remaining[:20] if len(remaining) > 20 else remaining
        prompt = f"""You are playing Wordle. Based on your experience:

Current state:
- Attempt {attempt + 1} of 6
- Remaining possible words ({len(remaining)}): {remaining_sample}
- Previous guesses and patterns: {history if history else 'None yet'}

You MUST choose ONE word from the remaining possibilities list above.
Respond with ONLY the chosen word in uppercase, nothing else.
"""
        
        try:
            client = ollama.Client(host='http://ollama.cs.wallawalla.edu:11434')
            response = client.generate(model='llama3.2', prompt=prompt)
            full_response = response['response'].strip()
            
            if verbose:
                print(f"LLM response: {full_response[:150]}...")
            
            # Try to extract a valid word from response
            # Check each line and each word in the response
            chosen_word = None
            for line in full_response.upper().split('\n'):
                # Remove punctuation and get words
                words = line.replace(',', ' ').replace('.', ' ').split()
                for word in words:
                    clean_word = ''.join(c for c in word if c.isalpha())
                    if clean_word in remaining:
                        chosen_word = clean_word
                        break
                if chosen_word:
                    break
            
            # Fallback: use highest entropy word from remaining
            if not chosen_word:
                if verbose:
                    print(f"  LLM output '{full_response[:50]}...' not in remaining words, using fallback...")
                # Use entropy to pick a good fallback
                if len(remaining) <= 50:
                    word_scores = [(w, calculate_pattern_entropy(w, remaining)) for w in remaining]
                    word_scores.sort(key=lambda x: x[1], reverse=True)
                    chosen_word = word_scores[0][0]
                else:
                    chosen_word = remaining[0]
            
            guesses.append(chosen_word)
            pattern = get_wordle_pattern(chosen_word, answer)
            
            if verbose:
                pattern_str = ''.join(['⬜' if p == 0 else '🟨' if p == 1 else '🟩' for p in pattern])
                print(f"\nAttempt {attempt + 1}: {chosen_word} -> {pattern_str}")
                print(f"Remaining: {len(remaining)}\n")
            
            if chosen_word == answer:
                if verbose:
                    print(f"✓ Solved in {len(guesses)} guesses!\n")
                return guesses
            
            history.append(f"{chosen_word} -> {pattern}")
            remaining = [w for w in remaining if apply_constraints(w, chosen_word, pattern)]
            
            if not remaining:
                if verbose:
                    print(f"✗ No valid words remain! Answer was {answer}\n")
                return guesses
            
        except Exception as e:
            if verbose:
                print(f"Error with Ollama: {e}")
                print(f"Falling back to entropy strategy...\n")
            # Fallback to entropy
            if remaining:
                if len(remaining) <= 50:
                    word_scores = [(w, calculate_pattern_entropy(w, remaining)) for w in remaining]
                    word_scores.sort(key=lambda x: x[1], reverse=True)
                    chosen_word = word_scores[0][0]
                else:
                    chosen_word = remaining[0]
                
                guesses.append(chosen_word)
                pattern = get_wordle_pattern(chosen_word, answer)
                
                if chosen_word == answer:
                    if verbose:
                        print(f"✓ Solved in {len(guesses)} guesses!\n")
                    return guesses
                
                remaining = [w for w in remaining if apply_constraints(w, chosen_word, pattern)]
    
    if verbose:
        print(f"✗ Failed to solve. Answer was {answer}\n")
    return guesses

# Demonstrate RL concept (with or without actual Ollama)
print("=== Reinforcement Learning Approach ===\n")
print("RL agents learn through experience rather than explicit rules.")
print("They discover strategies by:")
print("1. Trying actions in various states")
print("2. Receiving rewards (solved in fewer guesses = better)")
print("3. Updating their policy to prefer successful actions")
print("4. Balancing exploration (trying new things) vs exploitation (using what works)\n")

print("Modern LLMs like llama3.2 have implicitly 'learned' Wordle strategies")
print("from seeing many examples during training.\n")

# Try to run actual RL simulation if Ollama available
try:
    import ollama
    print("Ollama detected! Testing LLM-based agent...\n")
    
    # Use a small subset for LLM testing (faster and more reliable)
    random.seed(789)
    llm_word_list = random.sample(WORD_LIST, min(100, len(WORD_LIST)))
    test_answer = random.choice(llm_word_list)
    
    print(f"Testing with word list of {len(llm_word_list)} words")
    print(f"Target: {test_answer}\n")
    
    result = simulate_rl_agent_with_llm(test_answer, llm_word_list, verbose=True)
    
    if result:
        print(f"LLM agent completed in {len(result)} guesses: {' -> '.join(result)}")
    
except ImportError:
    print("Ollama not available - showing conceptual simulation only.")
    print("Install with: pip install ollama")
    print("Then run: ollama pull llama3.2\n")
except Exception as e:
    print(f"Could not connect to Ollama: {e}")
    print("Make sure Ollama is running with: ollama serve\n")

=== Reinforcement Learning Approach ===

RL agents learn through experience rather than explicit rules.
They discover strategies by:
1. Trying actions in various states
2. Receiving rewards (solved in fewer guesses = better)
3. Updating their policy to prefer successful actions
4. Balancing exploration (trying new things) vs exploitation (using what works)

Modern LLMs like llama3.2 have implicitly 'learned' Wordle strategies
from seeing many examples during training.

Ollama detected! Testing LLM-based agent...

Testing with word list of 100 words
Target: TANTI

=== Simulated RL Agent (via llama3.2) ===

LLM response: SNACK...

Attempt 1: SNACK -> ⬜🟨🟨⬜⬜
Remaining: 100

LLM response: ARNEE...

Attempt 2: ARNEE -> 🟨⬜🟩⬜⬜
Remaining: 7

LLM response: BUNDA...

Attempt 3: BUNDA -> ⬜⬜🟩⬜🟨
Remaining: 3

LLM response: TANTI...

Attempt 4: TANTI -> 🟩🟩🟩🟩🟩
Remaining: 1

✓ Solved in 4 guesses!

LLM agent completed in 4 guesses: SNACK -> ARNEE -> BUNDA -> TANTI


In [9]:
def analyze_difficult_cases(word_list):
    """Identify and test edge cases that challenge agents"""
    
    print("=== Failure Mode Analysis ===\n")
    
    # Case 1: Words with repeated letters
    repeated_letter_words = [w for w in word_list if len(set(w)) < 5]
    
    print(f"Case 1: Words with Repeated Letters")
    print(f"(Challenge: Less information per guess)\n")
    print(f"Found {len(repeated_letter_words)} words with repeated letters")
    
    if repeated_letter_words:
        # Test a few diverse examples
        test_words = random.sample(repeated_letter_words, min(3, len(repeated_letter_words)))
        print(f"Testing: {test_words}\n")
        
        agents = {
            'Frequency': frequency_agent_solve,
            'Entropy': entropy_agent_solve,
            'Minimax': minimax_agent_solve,
        }
        
        for test_word in test_words:
            unique_count = len(set(test_word))
            print(f"{test_word} ({unique_count} unique letters):")
            for name, agent in agents.items():
                guesses = agent(test_word, word_list, verbose=False)
                print(f"  {name}: {len(guesses)} guesses")
            print()
    
    # Case 2: Similar words (share many letters)
    print("\nCase 2: Word Clusters (Minimal Distinguishing Features)")
    print("(Challenge: Many words fit the same pattern)\n")
    
    # Find words with common prefixes/suffixes
    prefixes = defaultdict(list)
    for word in word_list:
        if len(word) >= 3:
            prefixes[word[:3]].append(word)
    
    large_clusters = [(prefix, words) for prefix, words in prefixes.items() if len(words) >= 5]
    large_clusters.sort(key=lambda x: len(x[1]), reverse=True)
    
    if large_clusters:
        prefix, cluster = large_clusters[0]
        print(f"Largest cluster with prefix '{prefix}': {len(cluster)} words")
        print(f"Sample: {cluster[:8]}\n")
        
        # Test on a couple from the cluster
        test_cluster = cluster[:3]
        for target in test_cluster:
            print(f"Target: {target}")
            for name, agent in {'Entropy': entropy_agent_solve,
                                'Minimax': minimax_agent_solve}.items():
                guesses = agent(target, word_list, verbose=False)
                print(f"  {name}: {len(guesses)} guesses")
            print()
    
    # Case 3: Uncommon letters
    print("\nCase 3: Words with Uncommon Letters")
    print("(Challenge: Less guided by letter frequency)\n")
    
    uncommon_letters = ['Q', 'X', 'Z', 'J']
    uncommon_words = [w for w in word_list if any(letter in w for letter in uncommon_letters)]
    
    if uncommon_words:
        print(f"Found {len(uncommon_words)} words with Q/X/Z/J")
        test_uncommon = random.sample(uncommon_words, min(3, len(uncommon_words)))
        print(f"Testing: {test_uncommon}\n")
        
        for target in test_uncommon:
            uncommon_in_word = [l for l in target if l in uncommon_letters]
            print(f"{target} (contains: {uncommon_in_word}):")
            for name in ['Frequency', 'Entropy']:
                agent = agents.get(name) or frequency_agent_solve if name == 'Frequency' else entropy_agent_solve
                guesses = agent(target, word_list, verbose=False)
                print(f"  {name}: {len(guesses)} guesses")
            print()

# Run failure analysis
print("Analyzing edge cases from full word list...\n")
analyze_difficult_cases(WORD_LIST)

Analyzing edge cases from full word list...

=== Failure Mode Analysis ===

Case 1: Words with Repeated Letters
(Challenge: Less information per guess)

Found 3599 words with repeated letters
Testing: ['BRIER', 'COHOL', 'SARUS']

BRIER (4 unique letters):
  Frequency: 4 guesses
  Entropy: 4 guesses
  Minimax: 4 guesses

COHOL (4 unique letters):
  Frequency: 4 guesses
  Entropy: 4 guesses
  Minimax: 4 guesses

SARUS (4 unique letters):
  Frequency: 4 guesses
  Entropy: 4 guesses
  Minimax: 6 guesses


Case 2: Word Clusters (Minimal Distinguishing Features)
(Challenge: Many words fit the same pattern)

Largest cluster with prefix 'CHA': 41 words
Sample: ['CHACK', 'CHACO', 'CHAFE', 'CHAFF', 'CHAFT', 'CHAGA', 'CHAIN', 'CHAIR']

Target: CHACK
  Entropy: 4 guesses
  Minimax: 3 guesses

Target: CHACO
  Entropy: 4 guesses
  Minimax: 4 guesses

Target: CHAFE
  Entropy: 4 guesses
  Minimax: 5 guesses


Case 3: Words with Uncommon Letters
(Challenge: Less guided by letter frequency)

Found 831 w